[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_09/MFM_ch9_ml_vs_har/MFM_ch9_ml_vs_har.ipynb)

# MFM_ch9_ml_vs_har

Case study, Christensen, Siggaard and Veliyev (2023, Journal of Financial Econometrics): out-of-sample MSE of HAR models, penalised regressions, regression trees and neural networks relative to HAR, averaged over 29 DJIA stocks (2001-2017), by forecast horizon (1 day, 1 week, 1 month) and information set (RV lags only vs RV lags plus 9 predictors); ranking of the 21 models one month ahead. Published numbers from Tables 2-7 of the paper.

*Modelling Financial Markets (MFM), Chapter 9: Realised Volatility. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_9'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# Colab: !pip install arch statsmodels
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Brand colours (reference lines in black; light bands only for intervals)
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Teal     = '#17A2B8'
Black    = '#000000'
LightGray = '#DADADA'
MODEL_COL = {'logHAR': IDAred, 'HAR': Orange, 'GARCH-t': MainBlue, 'EWMA': Purple, 'RW': Forest}
MODEL_LABEL = {'logHAR': 'log-HAR', 'HAR': 'HAR', 'GARCH-t': 'GARCH(1,1)-t', 'EWMA': 'EWMA', 'RW': "Yesterday's RV"}

HERE = '.'
SEED = 42
OOS_SPY = '2022-01-03'
OOS_BTC = '2025-03-01'
B_BOOT = 2000
C_START = '2025-03-01'


def save_fig(name):
    """Save the figure as a transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=0.0):
    """A single legend for a multi-panel figure, below the figure."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            h, l = ax.get_legend_handles_labels()
            for hh, ll in zip(h, l):
                if ll not in labels:
                    handles.append(hh)
                    labels.append(ll)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(x):
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in x]
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, pd.Timestamp):
        return str(x.date())
    return x


def ann(v, ppy):
    """Annualised volatility (%) from a daily variance (%^2)."""
    return np.sqrt(ppy * v)

## Helper functions

In [ ]:
CSV_MODELS = ['HAR-X', 'LogHAR', 'LevHAR', 'SHAR', 'HARQ', 'RR', 'LA', 'EN', 'A-LA', 'P-LA', 'BG', 'RF', 'GB', 'NN1_1', 'NN10_1', 'NN1_2', 'NN10_2', 'NN1_3', 'NN10_3', 'NN1_4', 'NN10_4']

CSV_REL_MSE = {('day', 'HAR'): (2, [1.0, 0.995, 1.073, 1.009, 1.059, 1.0, 1.003, 0.999, 1.007, 1.005, 1.147, 1.02, 1.054, 0.98, 0.969, 0.966, 0.958, 0.955, 0.954, 0.984, 0.99]), ('day', 'ALL'): (3, [0.966, 0.901, 1.003, 1.08, 1.289, 0.919, 0.936, 0.916, 0.957, 0.987, 0.961, 0.901, 0.962, 0.902, 0.889, 0.893, 0.885, 0.91, 0.898, 0.929, 0.944]), ('week', 'HAR'): (4, [1.0, 1.091, 1.089, 0.937, 1.043, 1.019, 1.024, 1.019, 1.033, 1.016, 1.056, 0.959, 1.164, 0.955, 0.951, 0.947, 0.937, 0.929, 0.925, 1.0, 1.011]), ('week', 'ALL'): (5, [1.037, 0.97, 1.076, 0.913, 1.459, 0.954, 0.958, 0.925, 1.026, 1.033, 0.84, 0.796, 0.93, 0.877, 0.857, 0.865, 0.846, 0.882, 0.855, 0.913, 0.945]), ('month', 'HAR'): (6, [1.0, 1.052, 1.03, 1.034, 0.994, 1.1, 1.107, 1.111, 1.118, 1.057, 1.171, 1.019, 1.219, 0.993, 0.97, 0.989, 0.958, 0.904, 0.905, 1.027, 1.068]), ('month', 'ALL'): (7, [1.29, 0.919, 1.328, 1.312, 1.569, 1.111, 1.378, 1.107, 1.386, 1.363, 0.621, 0.604, 0.829, 0.8, 0.789, 0.816, 0.786, 0.788, 0.795, 0.902, 0.964])}

CSV_LABEL = {'HAR-X': 'HAR-X (OLS, all predictors)', 'LogHAR': 'log-HAR', 'LevHAR': 'HAR with leverage', 'SHAR': 'SHAR', 'HARQ': 'HARQ', 'RR': 'Ridge', 'LA': 'Lasso', 'EN': 'Elastic net', 'A-LA': 'Adaptive lasso', 'P-LA': 'Post-lasso', 'BG': 'Bagging', 'RF': 'Random forest', 'GB': 'Gradient boosting'}

CSV_FAMILY = [('HAR family (OLS)', Orange, ['HAR-X', 'LogHAR', 'LevHAR', 'SHAR', 'HARQ']),
              ('Penalised regression', Purple, ['RR', 'LA', 'EN', 'A-LA', 'P-LA']),
              ('Regression trees', Forest, ['BG', 'RF', 'GB']),
              ('Neural networks', MainBlue, ['NN1_1', 'NN10_1', 'NN1_2', 'NN10_2', 'NN1_3', 'NN10_3', 'NN1_4', 'NN10_4'])]

## Analysis

In [ ]:
def csv_label(m):
    """Label of a model; NNe_k -> NN_k^e (notation of the paper)."""
    if m.startswith('NN'):
        e, k = m[2:].split('_')
        return rf'NN$_{k}^{{{e}}}$'
    return CSV_LABEL[m]


def csv_table():
    """Numbers from the paper: row = (horizon, information set), columns = models."""
    return pd.DataFrame({k: v[1] for k, v in CSV_REL_MSE.items()}, index=CSV_MODELS).T


def fig_csv_horizons():
    """MSE relative to HAR by horizon (day, week, month), for M_HAR and M_ALL (Tables 2-7)."""
    tab = csv_table()
    hz = ['day', 'week', 'month']
    show = [('HAR-X', Amber, 'D'), ('LogHAR', IDAred, 'o'), ('HARQ', Orange, 'v'), ('EN', Purple, 's'),
            ('RF', Forest, '^'), ('NN10_2', MainBlue, 'P')]
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.4), sharey=True)
    for ax, info, ttl in [(axes[0], 'HAR', r'$\mathcal{M}_{HAR}$: daily, weekly, monthly RV (Tables 2, 4, 6)'),
                          (axes[1], 'ALL', r'$\mathcal{M}_{ALL}$: RV lags + 9 predictors (12 in total; Tables 3, 5, 7)')]:
        ax.axhline(1.0, color=Black, lw=0.8, ls='--', label='HAR = 1')
        for m, c, mk in show:
            ax.plot(range(3), [tab.loc[(h, info), m] for h in hz], marker=mk, ms=5, color=c, lw=1.3, label=csv_label(m))
        ax.set_xticks(range(3))
        ax.set_xticklabels(['1 day', '1 week', '1 month'])
        ax.set_xlim(-0.25, 2.45)
        ax.set_title(ttl, fontsize=9.5)
        ax.set_xlabel('Forecast horizon')
    axes[0].set_ylabel('Out-of-sample MSE relative to HAR')
    for m in ['RF', 'NN10_2', 'HARQ', 'HAR-X']:
        v = tab.loc[('month', 'ALL'), m]
        axes[1].annotate(f'{v:.3f}', (2, v), xytext=(7, 0), textcoords='offset points', va='center', fontsize=8,
                         color='black')
    plt.tight_layout()
    fig_legend_bottom(fig, ncol=4, y=0.0)
    save_fig('ch9_csv_horizons')
    return {'rf_all': [tab.loc[(h, 'ALL'), 'RF'] for h in hz], 'nn_all': [tab.loc[(h, 'ALL'), 'NN10_2'] for h in hz]}


def fig_csv_ranking():
    """Ranking of the 21 models, one month ahead, M_ALL (Table 7), coloured by family."""
    s = csv_table().loc[('month', 'ALL')].sort_values(ascending=False)
    col = {m: c for _, c, ms in CSV_FAMILY for m in ms}
    fig, ax = plt.subplots(figsize=(6.6, 4.8))
    y = np.arange(len(s))
    ax.barh(y, s.values - 1.0, left=1.0, color=[col[m] for m in s.index], height=0.7)
    for yi, (m, v) in zip(y, s.items()):
        ax.text(v + (0.012 if v >= 1 else -0.012), yi, f'{v:.3f}', va='center', ha='left' if v >= 1 else 'right',
                fontsize=10, color='black')
    ax.axvline(1.0, color=Black, lw=0.8, ls='--')
    ax.set_yticks(y)
    ax.set_yticklabels([csv_label(m) for m in s.index], fontsize=10.5)
    ax.set_xlim(0.45, 1.78)
    ax.set_xlabel('Out-of-sample MSE relative to HAR (below 1: more accurate)', fontsize=11)
    ax.tick_params(axis='x', labelsize=10)
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for _, c, _ in CSV_FAMILY]
    fig.legend(handles, [f for f, _, _ in CSV_FAMILY], loc='upper center', bbox_to_anchor=(0.5, 0.01), ncol=4,
               frameon=False, fontsize=9.5, handlelength=1.2, columnspacing=1.0)
    plt.tight_layout(rect=(0, 0.03, 1, 1))
    save_fig('ch9_csv_ranking')
    return s.to_dict()

## Run

In [ ]:
print(fig_csv_horizons())
print(fig_csv_ranking())
csv_table().round(3)

![ch9_csv_horizons](./ch9_csv_horizons.png)

Output: `ch9_csv_horizons.pdf`

![ch9_csv_ranking](./ch9_csv_ranking.png)

Output: `ch9_csv_ranking.pdf`